# Benchmark: Folding Severity Sweep (Barrier Method)

Same magnitude + density sweeps as the SLSQP folding severity benchmark,
but using the penalty -> log-barrier L-BFGS-B solver.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import time
import numpy as np
import matplotlib.pyplot as plt

from dvfopt import jacobian_det2D, generate_random_dvf, scale_dvf
import laplacian
from dvfopt.core.iterative2d_barrier import iterative_2d_barrier
from benchmark_utils import (
    run_correction, plot_jac_heatmaps, plot_jdet_histograms,
)


## Part 1: Magnitude Sweep

In [ ]:
MAGNITUDES = [1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]
TARGET_SIZE = (40, 40)
SEED = 42


In [ ]:
mag_results = {}
print(f"{'Magnitude':>10s}  {'Neg init':>10s}  {'Neg final':>10s}  "
      f"{'Time (s)':>10s}  {'Min Jdet':>10s}  {'L2 Error':>10s}")
print("-" * 72)
for mag in MAGNITUDES:
    dvf = generate_random_dvf((3, 1, 5, 5), mag, SEED)
    dvf = scale_dvf(dvf, TARGET_SIZE)
    mag_results[mag] = run_correction(dvf, iterative_2d_barrier)
    r = mag_results[mag]
    status = "OK" if r["n_neg_final"] == 0 else "FAIL"
    print(f"{mag:>10.1f}  {r['n_neg_init']:>10d}  {r['n_neg_final']:>10d}  "
          f"{r['time']:>10.2f}  {r['min_jdet']:>+10.6f}  {r['l2_err']:>10.4f}  [{status}]")


### Magnitude sweep plots

In [ ]:
mags = list(mag_results.keys())
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
m_negs = [mag_results[m]["n_neg_init"] for m in mags]
m_negs_final = [mag_results[m]["n_neg_final"] for m in mags]
m_times = [mag_results[m]["time"] for m in mags]
m_l2s = [mag_results[m]["l2_err"] for m in mags]
m_success = [mag_results[m]["n_neg_final"] == 0 for m in mags]
m_min_init = [mag_results[m]["min_jdet_init"] for m in mags]
m_min_final = [mag_results[m]["min_jdet"] for m in mags]

ax = axes[0, 0]; ax.plot(mags, m_negs, "o-", color="tab:red")
ax.set_xlabel("Max magnitude"); ax.set_ylabel("Neg pixels (init)"); ax.set_title("Folding vs Magnitude"); ax.grid(True, alpha=0.3)
ax = axes[0, 1]; colors = ["tab:green" if s else "tab:red" for s in m_success]
ax.bar([str(m) for m in mags], m_times, color=colors); ax.set_xlabel("Magnitude"); ax.set_ylabel("Time (s)"); ax.set_title("Runtime")
ax = axes[0, 2]; ax.plot(m_negs, m_l2s, "s-", color="tab:orange")
ax.set_xlabel("Neg pixels"); ax.set_ylabel("L2"); ax.set_title("L2 vs Folding"); ax.grid(True, alpha=0.3)
ax = axes[1, 0]; x = np.arange(len(mags)); w = 0.35
ax.bar(x-w/2, m_min_init, w, label="Before", color="tab:red", alpha=0.7)
ax.bar(x+w/2, m_min_final, w, label="After", color="tab:blue", alpha=0.7)
ax.axhline(0, color="black", linewidth=0.5); ax.set_xticks(x)
ax.set_xticklabels([str(m) for m in mags], rotation=30); ax.set_ylabel("Min Jdet"); ax.set_title("Worst-case Jdet"); ax.legend(fontsize=8)
ax = axes[1, 1]; ax.plot(mags, m_negs, "o--", color="tab:red", label="Before", alpha=0.6)
ax.plot(mags, m_negs_final, "s-", color="tab:blue", label="After", linewidth=2)
ax.set_xlabel("Magnitude"); ax.set_ylabel("Neg-Jdet count"); ax.set_title("Convergence"); ax.legend()
ax = axes[1, 2]; tpn = [mag_results[m]["time"]/max(mag_results[m]["n_neg_init"],1) for m in mags]
ax.bar([str(m) for m in mags], tpn, color="tab:purple"); ax.set_ylabel("Time/neg pixel"); ax.set_title("Cost per pixel")
plt.suptitle("Magnitude Sweep - barrier", fontsize=14); plt.tight_layout(); plt.show()


In [ ]:
plot_jac_heatmaps(
    [[mag_results[m]["jac_init"][0] for m in mags],
     [mag_results[m]["jac_final"][0] for m in mags]],
    [f"mag={m}" for m in mags],
    title="Jdet Before vs After (Magnitude, Barrier)",
)
plt.show()


In [ ]:
plot_jdet_histograms(
    [[("Before", mag_results[m]["jac_init"][0]),
      ("After", mag_results[m]["jac_final"][0])] for m in mags],
    [f"mag={m}" for m in mags],
    title="Jdet Distribution (Magnitude, Barrier)",
)
plt.show()


## Part 2: Folding Density Sweep

In [ ]:
def make_crossing_pairs(n_pairs, H, W, seed=0):
    rng = np.random.default_rng(seed)
    margin = 3
    msample, fsample = [], []
    for _ in range(n_pairs):
        y1 = rng.integers(margin, H - margin)
        x1 = rng.integers(margin, W - margin)
        dy = rng.integers(2, 5); dx = rng.integers(2, 5)
        y2 = min(y1 + dy, H - margin - 1)
        x2 = min(x1 + dx, W - margin - 1)
        msample.append([0, y1, x1]); msample.append([0, y2, x2])
        fsample.append([0, y2, x2]); fsample.append([0, y1, x1])
    return np.array(msample), np.array(fsample)

PAIR_COUNTS = [1, 2, 4, 6, 8, 12, 16, 20]
GRID_H, GRID_W = 30, 30


In [ ]:
density_results = {}
print(f"{'Pairs':>8s}  {'Neg init':>10s}  {'Neg final':>10s}  {'Time (s)':>10s}  {'Min Jdet':>10s}  {'L2 Error':>10s}")
print("-" * 68)
for n_pairs in PAIR_COUNTS:
    ms, fs = make_crossing_pairs(n_pairs, GRID_H, GRID_W, seed=n_pairs)
    fixed_sample = np.zeros((1, GRID_H, GRID_W))
    dvf = laplacian.solveLaplacianFromCorrespondences(fixed_sample.shape, ms, fs)
    density_results[n_pairs] = run_correction(dvf, iterative_2d_barrier)
    r = density_results[n_pairs]
    status = "OK" if r["n_neg_final"] == 0 else "FAIL"
    print(f"{n_pairs:>8d}  {r['n_neg_init']:>10d}  {r['n_neg_final']:>10d}  "
          f"{r['time']:>10.2f}  {r['min_jdet']:>+10.6f}  {r['l2_err']:>10.4f}  [{status}]")


In [ ]:
pairs = list(density_results.keys())
plot_jac_heatmaps(
    [[density_results[p]["jac_init"][0] for p in pairs],
     [density_results[p]["jac_final"][0] for p in pairs]],
    [f"{p} pairs" for p in pairs],
    title="Jdet Before vs After (Density, Barrier)",
)
plt.show()
